# Least squares, inference and prediction: lab notebook

**Statistical Modelling and Forecasting, Regression and exponential smoothing**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

### 1. Quarterly growth of US consumption (y) against growth of disposable income (x), in percent

In [ ]:
# 1. Quarterly growth of US consumption (y) against growth of disposable income (x), in percent
d = sm.datasets.macrodata.load_pandas().data
g = 100 * np.log(d[["realcons", "realdpi"]]).diff().dropna()
x, y = g["realdpi"].values, g["realcons"].values
X = sm.add_constant(x)
fit = sm.OLS(y, X).fit()
print("n =", len(y), "| b0 =", round(fit.params[0], 3), "| b1 =", round(fit.params[1], 3))
print("se(b1) =", round(fit.bse[1], 4), "| t =", round(fit.tvalues[1], 2), "| p =", round(fit.pvalues[1], 4), "| R2 =", round(fit.rsquared, 3))
print("95% CI for the slope:", np.round(fit.conf_int()[1], 3), "| F =", round(fit.fvalue, 2), "= t^2 =", round(fit.tvalues[1] ** 2, 2))

### 2. The same numbers from the formulas

In [ ]:
# 2. The same numbers from the formulas
xb, yb = x.mean(), y.mean()
Sxx = ((x - xb) ** 2).sum()
b1 = ((x - xb) * (y - yb)).sum() / Sxx
s2 = ((y - fit.fittedvalues) ** 2).sum() / (len(y) - 2)
print("by formula: b1 =", round(b1, 3), "| sigma^2 =", round(s2, 4), "| se(b1) =", round(np.sqrt(s2 / Sxx), 4))

### 3. Confidence interval for the mean response and prediction interval for a new quarter

In [ ]:
# 3. Confidence interval for the mean response and prediction interval for a new quarter
x0 = 1.5
pr = fit.get_prediction(np.array([[1, x0]])).summary_frame(alpha=0.05)
print(pr.round(3).to_string(index=False))
tc = stats.t.ppf(0.975, len(y) - 2)
lev = 1 / len(y) + (x0 - xb) ** 2 / Sxx
print("half-width, mean:", round(tc * np.sqrt(s2 * lev), 3), "| new observation:", round(tc * np.sqrt(s2 * (1 + lev)), 3))

### 4. Intervals widen away from the mean of x

In [ ]:
# 4. Intervals widen away from the mean of x
for q in (xb, 1.5, 4.0):
    pp = fit.get_prediction(np.array([[1, q]])).summary_frame(alpha=0.05)
    print(f"x0 = {q:5.2f}  mean CI width = {pp.iloc[0, 3] - pp.iloc[0, 2]:.3f}   prediction interval width = {pp.iloc[0, 5] - pp.iloc[0, 4]:.3f}")

### 5. The numerical problem by hand: six points

In [ ]:
# 5. The numerical problem by hand: six points
xs = np.array([2, 4, 6, 8, 10, 12.0]); ys = np.array([9, 14, 16, 21, 23, 29.0])
m = sm.OLS(ys, sm.add_constant(xs)).fit()
print("slope", round(m.params[1], 3), "intercept", round(m.params[0], 3), "sigma^2", round(m.scale, 3), "t", round(m.tvalues[1], 2), "R2", round(m.rsquared, 4))
pp = m.get_prediction(np.array([[1, 14.0]])).summary_frame()
print("forecast at x0 = 14:", round(pp["mean"].iloc[0], 3), "| 95% PI half-width:", round((pp["obs_ci_upper"] - pp["obs_ci_lower"]).iloc[0] / 2, 3))

## Expected output

Running all cells above should print:

```
n = 202 | b0 = 0.555 | b1 = 0.341
se(b1) = 0.0493 | t = 6.91 | p = 0.0 | R2 = 0.193
95% CI for the slope: [0.244 0.438] | F = 47.79 = t^2 = 47.79
by formula: b1 = 0.341 | sigma^2 = 0.3911 | se(b1) = 0.0493
 mean  mean_se  mean_ci_lower  mean_ci_upper  obs_ci_lower  obs_ci_upper
1.066    0.055          0.957          1.175        -0.172         2.304
half-width, mean: 0.109 | new observation: 1.238
x0 =  0.83  mean CI width = 0.174   prediction interval width = 2.472
x0 =  1.50  mean CI width = 0.217   prediction interval width = 2.476
x0 =  4.00  mean CI width = 0.641   prediction interval width = 2.548
slope 1.886 intercept 5.467 sigma^2 1.105 t 15.01 R2 0.9826
forecast at x0 = 14: 31.867 | 95% PI half-width: 3.987
```

## Practice exercises

Try these in the cells below.

1. Increase the noise σ and watch R², the t-statistic and the width of the bands.
2. Move the prediction point x₀ to the far right. Compare the confidence band for the mean with the prediction interval.
3. Set the true slope to 0 and press New sample ten times. How often does the t-test reject at 5%?

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
